In [8]:
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
import pandas as pd
import numpy as np
from pathlib import Path

# Load all US counties, reproject to match raster
counties = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip")
counties = counties.to_crs("EPSG:5070")

# Keep only CONUS
conus = counties[~counties["STATEFP"].isin(["02", "15", "60", "66", "69", "72", "78"])]

state_fips = {
    "01":"AL","04":"AZ","05":"AR","06":"CA","08":"CO","09":"CT",
    "10":"DE","11":"DC","12":"FL","13":"GA","16":"ID","17":"IL",
    "18":"IN","19":"IA","20":"KS","21":"KY","22":"LA","23":"ME",
    "24":"MD","25":"MA","26":"MI","27":"MN","28":"MS","29":"MO",
    "30":"MT","31":"NE","32":"NV","33":"NH","34":"NJ","35":"NM",
    "36":"NY","37":"NC","38":"ND","39":"OH","40":"OK","41":"OR",
    "42":"PA","44":"RI","45":"SC","46":"SD","47":"TN","48":"TX",
    "49":"UT","50":"VT","51":"VA","53":"WA","54":"WV","55":"WI","56":"WY"
}

pixel_area_ha = 250 * 250 / 10000  # 6.25 ha

# Find all surplus tif files
surplus_dir = Path("../../Surplus")
tif_files = sorted(surplus_dir.glob("Surplus_N_*.tif"))
print(f"Found {len(tif_files)} files: {[f.name for f in tif_files]}")

# Loop over years
all_years = []

for tif_path in tif_files:
    year = int(tif_path.stem.split("_")[-1])  # extract year from filename
    print(f"Processing {year}...")

    with rasterio.open(tif_path) as src:
        nodata = src.nodata  # use file's nodata if set, else None

    stats = zonal_stats(
        conus,
        str(tif_path),
        stats=["mean", "sum", "count"],
        nodata=nodata,
        geojson_out=False
    )

    df = pd.DataFrame(stats)
    df["year"]     = year
    df["GEOID"]    = conus["GEOID"].values
    df["STATEFP"]  = conus["STATEFP"].values
    df["COUNTYFP"] = conus["COUNTYFP"].values
    df["NAME"]     = conus["NAME"].values
    df["NAMELSAD"] = conus["NAMELSAD"].values
    df["STATE"]    = df["STATEFP"].map(state_fips)

    df["total_kg_N"] = df["sum"] * pixel_area_ha

    all_years.append(df)

# Combine into panel
panel = pd.concat(all_years, ignore_index=True)
panel = panel.rename(columns={"mean": "mean_surplus_kgha", "sum": "raw_sum", "count": "pixel_count"})
panel = panel[["year", "GEOID", "STATEFP", "STATE", "COUNTYFP", "NAME", "NAMELSAD",
               "mean_surplus_kgha", "raw_sum", "total_kg_N", "pixel_count"]]

panel.to_csv("nitrogen_surplus_by_county_panel.csv", index=False)
print(f"Done. Shape: {panel.shape}")
print(panel.head())

Found 88 files: ['Surplus_N_1930.tif', 'Surplus_N_1931.tif', 'Surplus_N_1932.tif', 'Surplus_N_1933.tif', 'Surplus_N_1934.tif', 'Surplus_N_1935.tif', 'Surplus_N_1936.tif', 'Surplus_N_1937.tif', 'Surplus_N_1938.tif', 'Surplus_N_1939.tif', 'Surplus_N_1940.tif', 'Surplus_N_1941.tif', 'Surplus_N_1942.tif', 'Surplus_N_1943.tif', 'Surplus_N_1944.tif', 'Surplus_N_1945.tif', 'Surplus_N_1946.tif', 'Surplus_N_1947.tif', 'Surplus_N_1948.tif', 'Surplus_N_1949.tif', 'Surplus_N_1950.tif', 'Surplus_N_1951.tif', 'Surplus_N_1952.tif', 'Surplus_N_1953.tif', 'Surplus_N_1954.tif', 'Surplus_N_1955.tif', 'Surplus_N_1956.tif', 'Surplus_N_1957.tif', 'Surplus_N_1958.tif', 'Surplus_N_1959.tif', 'Surplus_N_1960.tif', 'Surplus_N_1961.tif', 'Surplus_N_1962.tif', 'Surplus_N_1963.tif', 'Surplus_N_1964.tif', 'Surplus_N_1965.tif', 'Surplus_N_1966.tif', 'Surplus_N_1967.tif', 'Surplus_N_1968.tif', 'Surplus_N_1969.tif', 'Surplus_N_1970.tif', 'Surplus_N_1971.tif', 'Surplus_N_1972.tif', 'Surplus_N_1973.tif', 'Surplus_N_1974

/Users/rkaur26/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/rasterstats/io.py:437: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


Processing 1931...
Processing 1932...
Processing 1933...
Processing 1934...
Processing 1935...
Processing 1936...
Processing 1937...
Processing 1938...
Processing 1939...
Processing 1940...
Processing 1941...
Processing 1942...
Processing 1943...
Processing 1944...
Processing 1945...
Processing 1946...
Processing 1947...
Processing 1948...
Processing 1949...
Processing 1950...
Processing 1951...
Processing 1952...
Processing 1953...
Processing 1954...
Processing 1955...
Processing 1956...
Processing 1957...
Processing 1958...
Processing 1959...
Processing 1960...
Processing 1961...
Processing 1962...
Processing 1963...
Processing 1964...
Processing 1965...
Processing 1966...
Processing 1967...
Processing 1968...
Processing 1969...
Processing 1970...
Processing 1971...
Processing 1972...
Processing 1973...
Processing 1974...
Processing 1975...
Processing 1976...
Processing 1977...
Processing 1978...
Processing 1979...
Processing 1980...
Processing 1981...
Processing 1982...
Processing 1

In [9]:
df.head()

,mean,count,sum,year,GEOID,STATEFP,COUNTYFP,NAME,NAMELSAD,STATE,total_kg_N
0,198.377184,23812,4.723758e+06,2017,31039,31,039,Cuming,Cuming County,NE,2.952348e+07
1,3.326338,11456,3.810653e+04,2017,53069,53,069,Wahkiakum,Wahkiakum County,WA,2.381658e+05
2,2.637125,96736,2.551049e+05,2017,35011,35,011,De Baca,De Baca County,NM,1.594406e+06
3,39.153392,35080,1.373501e+06,2017,31109,31,109,Lancaster,Lancaster County,NE,8.584381e+06
4,48.974322,23858,1.168429e+06,2017,31129,31,129,Nuckolls,Nuckolls County,NE,7.302684e+06
